In [15]:
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report


In [16]:
df_train = pd.read_csv(r"D:\SER_Cross\data\processed\train.csv")
df_val   = pd.read_csv(r"D:\SER_Cross\data\processed\val.csv")
df_test  = pd.read_csv(r"D:\SER_Cross\data\processed\test.csv")


In [17]:
SAMPLE_RATE = 22050
DURATION = 4
N_MELS = 128
MAX_LEN = 173

def spec_augment(mel):
    mel = mel.copy()
    t = mel.shape[1]
    t0 = np.random.randint(0, t - 20)
    mel[:, t0:t0+20] = 0
    return mel

def extract_logmel(path, augment=False):
    y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION)

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        n_fft=1024,
        hop_length=512
    )

    logmel = librosa.power_to_db(mel)

    # 🔥 CRITICAL NORMALIZATION
    logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)

    if augment:
        logmel = spec_augment(logmel)

    if logmel.shape[1] < MAX_LEN:
        logmel = np.pad(logmel, ((0,0),(0,MAX_LEN-logmel.shape[1])))
    else:
        logmel = logmel[:, :MAX_LEN]

    return logmel.T   # ⬅ (time, mel)


In [18]:
X_train = np.array([
    extract_logmel(p, augment=True)
    for p in df_train["path"]
])

X_val = np.array([
    extract_logmel(p, augment=False)
    for p in df_val["path"]
])

X_test = np.array([
    extract_logmel(p, augment=False)
    for p in df_test["path"]
])


In [19]:
le = LabelEncoder()
le.fit(df_train["emotion"])

y_train = le.transform(df_train["emotion"])
y_val   = le.transform(df_val["emotion"])
y_test  = le.transform(df_test["emotion"])


In [20]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)


Class Weights: {0: 0.9859922178988327, 1: 0.9859922178988327, 2: 0.9859922178988327, 3: 1.0602510460251047, 4: 0.9859922178988327}


In [21]:
model = tf.keras.Sequential([
    layers.Conv1D(64, 5, activation="relu", input_shape=(173,128)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),

    layers.Conv1D(128, 5, activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),

    layers.Bidirectional(layers.LSTM(96)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
])


c:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_4 (Conv1D)               │ (None, 169, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 169, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 84, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 80, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 80, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 40, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 192)            │       172,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 281,029 (1.07 MB)

 Trainable params: 280,645 (1.07 MB)

 Non-trainable params: 384 (1.50 KB)

In [23]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    class_weight=class_weights
)


Epoch 1/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.4191 - loss: 1.3413 - val_accuracy: 0.3977 - val_loss: 1.4127
Epoch 2/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.4974 - loss: 1.2164 - val_accuracy: 0.4462 - val_loss: 1.3947
Epoch 3/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.5233 - loss: 1.1290 - val_accuracy: 0.4801 - val_loss: 1.3457
Epoch 4/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5578 - loss: 1.0730 - val_accuracy: 0.4869 - val_loss: 1.3256
Epoch 5/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5902 - loss: 1.0040 - val_accuracy: 0.4636 - val_loss: 1.4686
Epoch 6/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6176 - loss: 0.9443 - val_accuracy: 0.4753 - val_loss: 1.3500
Epoch 7/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6425 - loss: 0.8996 - val_accuracy: 0.4685 - val_loss: 1.5753
Epoch 8/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6661 - loss: 0.8433 - val_accu

In [24]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Final CRNN Test Accuracy:", test_acc)


36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5807 - loss: 2.5881
Final CRNN Test Accuracy: 0.5806737542152405


In [25]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))


36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
              precision    recall  f1-score   support

       angry       0.63      0.89      0.74       228
        fear       0.54      0.32      0.41       228
       happy       0.51      0.39      0.45       228
     neutral       0.59      0.52      0.55       216
         sad       0.58      0.77      0.66       228

    accuracy                           0.58      1128
   macro avg       0.57      0.58      0.56      1128
weighted avg       0.57      0.58      0.56      1128

